# 01. 수집 — 리뷰 API와 업적 API

**출력**: `raw/reviews.jsonl`(API 응답 원본), `starfield_reviews.csv`, `raw/ach.jsonl`, `starfield_achievements.csv`, `starfield_analysis.csv`

## #A 리뷰 전수 수집

Steam 리뷰 API(`store.steampowered.com/appreviews/<게임id>`)는 한 번에 100건씩, 커서를 따라가며 준다.
그런데 한 조회 조건에서 약 2,000건까지만 주고 커서가 멈춘다(실측). "9월 리뷰 다 줘"라고 하면 90,178건 중 1,600건만 받는다.

그래서 **기간을 잘게 쪼갠다**: 먼저 건수만 물어보고(요청 1번), 1,500건을 넘으면 기간을 반으로 잘라 다시 물어본다.
끝낸 구간은 `raw/done.jsonl`에 기록해 두어, 중간에 끊겨도 다시 실행하면 이어서 받는다.

In [ ]:
import requests
import json
import time
import os

APPID = 1716740                       # Starfield
BASE = 'https://store.steampowered.com'
SAFE = 1500                           # 한 구간에서 이 건수를 넘으면 기간을 쪼갠다
DELAY = 0.5                           # 요청 사이 대기(초)
START = 1690848000                    # 2023-08-01 00:00 UTC
END = 1788825600                      # 2026-09-08 00:00 UTC

os.makedirs('raw', exist_ok=True)
session = requests.Session()


def ask(params):
    """리뷰 API를 한 번 호출해 JSON 응답을 돌려준다. params에 start_date, end_date, cursor 등을 넣는다."""
    base_params = {'json': 1, 'filter': 'recent', 'language': 'all', 'purchase_type': 'all',
                   'review_type': 'all', 'date_range_type': 'include'}
    base_params.update(params)
    response = session.get(f'{BASE}/appreviews/{APPID}', params=base_params, timeout=30)
    return response.json()


def count(start, end):
    """구간 안의 리뷰 건수와 긍정 건수만 물어본다 (num_per_page=0이면 리뷰 본문은 안 온다)."""
    r = ask({'num_per_page': 0, 'start_date': start, 'end_date': end})
    return r['query_summary']


def fetch(start, end, out_file):
    """구간 안의 리뷰를 커서를 따라가며 전부 받아 파일에 한 줄씩 쓴다. 받은 건수를 돌려준다."""
    cursor = '*'
    seen = set()          # 같은 리뷰가 두 번 오면 건너뛰기 위한 id 모음
    while True:
        r = ask({'num_per_page': 100, 'cursor': cursor, 'start_date': start, 'end_date': end})
        reviews = r.get('reviews', [])
        if len(reviews) == 0:
            break
        for review in reviews:
            rid = review['recommendationid']
            if rid in seen:
                continue
            seen.add(rid)
            out_file.write(json.dumps(review, ensure_ascii=False) + '\n')
        next_cursor = r.get('cursor')
        if next_cursor is None or next_cursor == cursor:
            break
        cursor = next_cursor
        time.sleep(DELAY)
    return len(seen)

In [ ]:
# 이미 끝낸 구간 읽기 (재실행 대비)
done = set()
if os.path.exists('raw/done.jsonl'):
    for line in open('raw/done.jsonl'):
        w = json.loads(line)['w']
        done.add((w[0], w[1]))

# 처리할 구간을 쌓아 두는 목록. 꺼내서 건수를 보고, 많으면 반으로 쪼개 다시 넣는다
todo = [(START, END)]
total = 0
short = []      # 끝까지 못 받은 구간

out_file = open('raw/reviews.jsonl', 'a', encoding='utf-8')
log_file = open('raw/done.jsonl', 'a', encoding='utf-8')

while len(todo) > 0:
    start, end = todo.pop()
    if (start, end) in done:
        continue

    summary = count(start, end)
    n = summary['total_reviews']

    # 건수가 많고 구간이 1초보다 길면 반으로 쪼갠다
    if n > SAFE and end - start > 1:
        mid = (start + end) // 2
        todo.append((mid, end))
        todo.append((start, mid))
        continue

    if n > 0:
        got = fetch(start, end, out_file)
    else:
        got = 0

    # 받은 건수가 기대의 98%에 못 미치면 더 쪼개서 다시 시도
    if got < n * 0.98 and end - start > 1:
        mid = (start + end) // 2
        todo.append((mid, end))
        todo.append((start, mid))
        continue
    if got < n * 0.98:
        short.append((start, end, n, got))

    total = total + got
    log_file.write(json.dumps({'w': [start, end], 'n': n, 'pos': summary['total_positive']}) + '\n')
    log_file.flush()
    done.add((start, end))
    print(f'\r수집 {total:,}건 · 남은 구간 {len(todo)}   ', end='')

out_file.close()
log_file.close()
print(f'\n완료: {total:,}건')
if len(short) > 0:
    print('[!] 끝까지 못 받은 구간:', short)

In [ ]:
# JSON 원본 → 표(CSV)로 바꾸기
import pandas as pd

rows = []
for line in open('raw/reviews.jsonl', encoding='utf-8'):
    rows.append(json.loads(line))

df = pd.json_normalize(rows)                     # 중첩된 author.* 필드를 열로 펼친다
df = df.drop_duplicates('recommendationid')

df = df.rename(columns={
    'author.steamid': 'steamid',
    'author.num_games_owned': 'games_owned',
    'author.num_reviews': 'num_reviews',
    'author.playtime_forever': 'playtime_forever_min',
    'author.playtime_at_review': 'playtime_at_review_min',
    'author.playtime_last_two_weeks': 'playtime_2w_min',
    'author.last_played': 'last_played',
    'timestamp_created': 'ts_created',
    'timestamp_updated': 'ts_updated',
})
df['steamid'] = df['steamid'].astype('string')
df['weighted_vote_score'] = pd.to_numeric(df['weighted_vote_score'], errors='coerce')
df['review'] = df['review'].str.replace(r'\s+', ' ', regex=True).str.strip()   # 연속 공백·줄바꿈을 한 칸으로
df['month'] = pd.to_datetime(df['ts_created'], unit='s').dt.to_period('M').astype(str)

df.to_json('starfield_reviews.jsonl', orient='records', lines=True, force_ascii=False)
df.to_csv('starfield_reviews.csv', index=False)

# API가 말하는 전체 건수와 비교
summary = count(START, END)
print('기대', summary['total_reviews'], '/ 수집', len(df), '=', round(len(df) / summary['total_reviews'] * 100, 2), '%')
print('기대 긍정', summary['total_positive'], '/ 실제 긍정', int(df['voted_up'].sum()))
print('기간', df['month'].min(), '~', df['month'].max(), '· 긍정률', round(df['voted_up'].mean() * 100, 2), '%')

In [ ]:
# 구간 기록에 빈틈이 없는지 확인: 구간들을 정렬했을 때 앞 구간의 끝 = 뒤 구간의 시작이어야 한다
windows = []
for line in open('raw/done.jsonl'):
    windows.append(json.loads(line)['w'])
windows.sort()

gaps = 0
for i in range(len(windows) - 1):
    if windows[i][1] != windows[i + 1][0]:
        gaps = gaps + 1
print(len(windows), '구간 · 시작 일치', windows[0][0] == START, '· 끝 일치', windows[-1][1] == END, '· 빈틈', gaps)

## #B 업적 공개 유저 수집

Steam Web API 키가 필요하다. 실행할 때 입력받고 코드나 파일에는 남지 않는다.

업적은 한 사람에 한 요청이라 비싸다. 그런데 대부분은 프로필 비공개라 헛수고다.
프로필 요약은 100명씩 묶어 조회할 수 있으므로 **먼저 공개자만 걸러낸 뒤** 업적을 조회한다.

In [ ]:
import getpass
from concurrent.futures import ThreadPoolExecutor
import threading

KEY = getpass.getpass('Steam API Key: ')
API = 'https://api.steampowered.com'

rev = pd.read_csv('starfield_reviews.csv', dtype={'steamid': 'string'}, low_memory=False)
ids = rev['steamid'].dropna().unique().tolist()
print('리뷰어', len(ids), '명')

In [ ]:
# 1단계: 프로필 공개 여부 (100명씩 한 요청). 진행 상황을 파일에 저장해 끊겨도 이어서 한다
progress_path = 'raw/public_ids.json'
if os.path.exists(progress_path):
    state = json.load(open(progress_path))
else:
    state = {'i': 0, 'public': []}
public = state['public']

for i in range(state['i'], len(ids), 100):
    chunk = ids[i:i + 100]
    while True:
        response = session.get(f'{API}/ISteamUser/GetPlayerSummaries/v2/',
                               params={'key': KEY, 'steamids': ','.join(chunk)}, timeout=30)
        if response.ok:
            break
        print(f'\rHTTP {response.status_code} → 5분 대기 후 재시도   ', end='')
        time.sleep(300)

    for player in response.json()['response']['players']:
        if player.get('communityvisibilitystate') == 3:     # 3 = 공개
            public.append(player['steamid'])

    if (i // 100) % 20 == 0:                                  # 20요청마다 진행 상황 저장
        json.dump({'i': i + 100, 'public': public}, open(progress_path, 'w'))
    time.sleep(1)
    print(f'\r{i + len(chunk):,}/{len(ids):,} · 공개 {len(public):,}   ', end='')

json.dump({'i': len(ids), 'public': public}, open(progress_path, 'w'))
print(f'\n프로필 공개 {len(public):,}명 ({len(public) / len(ids) * 100:.1f}%)')

In [ ]:
# 2단계: 공개자의 업적 조회 (1인 1요청, 8개 동시 실행). 결과는 실패도 기록해 재실행 때 건너뛴다
done = set()
if os.path.exists('raw/ach.jsonl'):
    for line in open('raw/ach.jsonl'):
        done.add(json.loads(line)['steamid'])

todo = []
for sid in public:
    if sid not in done:
        todo.append(sid)
todo = todo[:90000]         # 하루 요청 제한(10만) 안에서
print('남은', len(todo), '명')

lock = threading.Lock()
out_file = open('raw/ach.jsonl', 'a', encoding='utf-8')


def get_achievements(sid):
    """한 사람의 업적을 조회해 파일에 한 줄 쓴다. 성공하면 1, 실패하면 0."""
    for attempt in range(2):
        try:
            response = requests.get(f'{API}/ISteamUserStats/GetPlayerAchievements/v1/',
                                    params={'key': KEY, 'appid': APPID, 'steamid': sid, 'l': 'english'}, timeout=30)
            stats = response.json()['playerstats']
        except Exception:
            time.sleep(2)
            continue

        ok = bool(stats.get('success'))
        achievements = []
        if ok:
            for a in stats.get('achievements', []):
                if a['achieved'] == 1 and a['unlocktime'] > 0:
                    achievements.append({'name': a['name'], 't': a['unlocktime']})

        with lock:      # 여러 스레드가 동시에 파일에 쓰지 않도록
            out_file.write(json.dumps({'steamid': sid, 'ok': ok, 'ach': achievements}, ensure_ascii=False) + '\n')
        return int(ok)
    return 0


ok_count = 0
with ThreadPoolExecutor(max_workers=8) as executor:
    for i, result in enumerate(executor.map(get_achievements, todo), 1):
        ok_count = ok_count + result
        if i % 200 == 0:
            print(f'\r{i:,}/{len(todo):,} · 성공 {ok_count:,} ({ok_count / i * 100:.1f}%)   ', end='')
out_file.close()
print(f'\n업적 확보 {ok_count:,}명')

## 분석 테이블 만들기 — 리뷰 + "리뷰 시점의 진행 단계"

메인 퀘스트 업적 10개에 진행 순서대로 1~10 번호를 붙인다.
리뷰를 쓰기 **전에** 해금한 업적만 모아, 그중 가장 뒤 단계를 `stage_at_review`로 삼는다.

In [ ]:
MAIN = ['For All, Into the Starfield', 'One Small Step', 'Into the Unknown',
        'All That Money Can Buy', 'Further Into the Unknown', 'High Price to Pay',
        'In Their Footsteps', 'Entangled', 'Unearthed', 'One Giant Leap']
STAGE = {}
for i, name in enumerate(MAIN):
    STAGE[name] = i + 1

# 업적 파일 → 사람 목록(players)과 (사람, 업적, 시각) 행 목록(rows)
players = []
rows = []
for line in open('raw/ach.jsonl', encoding='utf-8'):
    d = json.loads(line)
    if not d['ok']:
        continue
    players.append(d['steamid'])
    for a in d['ach']:
        rows.append((d['steamid'], a['name'], a['t']))

ev = pd.DataFrame(rows, columns=['steamid', 'name', 't'])
ev['steamid'] = ev['steamid'].astype('string')
ev['stage'] = ev['name'].map(STAGE)          # 메인 퀘스트가 아니면 NaN

# 리뷰어당 마지막 리뷰 1건만, 업적 조회에 성공한 사람만
tbl = rev.sort_values('ts_created').drop_duplicates('steamid', keep='last')
tbl = tbl[tbl['steamid'].isin(players)]

# 업적에 리뷰 작성 시각을 붙여, 리뷰 전에 해금한 업적만 고른다
ev = ev.merge(tbl[['steamid', 'ts_created']], on='steamid')
before = ev[ev['t'] < ev['ts_created']]

# 사람별 집계: 최종 단계·총 업적 수 (전체) / 리뷰 시점 단계·업적 수 (리뷰 전)
total_stats = ev.groupby('steamid').agg(stage_final=('stage', 'max'), n_ach_total=('t', 'size'),
                                         first_unlock=('t', 'min'), last_unlock=('t', 'max'))
review_stats = before.groupby('steamid').agg(stage_at_review=('stage', 'max'), n_ach_at_review=('t', 'size'))

tbl = tbl.merge(total_stats, on='steamid', how='left')
tbl = tbl.merge(review_stats, on='steamid', how='left')

for col in ['stage_at_review', 'stage_final', 'n_ach_at_review', 'n_ach_total']:
    tbl[col] = tbl[col].fillna(0).astype(int)
tbl['gap_days'] = (tbl['last_played'] - tbl['last_unlock']) / 86400

tbl.to_csv('starfield_analysis.csv', index=False)
ev.to_csv('starfield_achievements.csv', index=False)
print('분석 대상', len(tbl), '명')
print('해금 0건 비율', round((tbl['n_ach_total'] == 0).mean() * 100, 1), '%')
print(tbl['stage_at_review'].value_counts().sort_index())

## #C 사전 조사 — 후보 게임 4개 비교

계획 단계에서 같은 수집기로 Starfield · RDR2(1174180) · Cyberpunk 2077(1091500) · Diablo IV(2344520)의 리뷰를 모아
전체 긍정률과 **리뷰 시점 플레이타임 구간별 긍정률**을 비교했다. 재현하려면 #A의 `APPID`를 바꿔 수집한 뒤 아래 함수를 쓴다.

In [ ]:
def positivity_by_playtime(reviews):
    """리뷰 시점 플레이타임(시간) 구간별 리뷰 수와 긍정률."""
    hours = reviews['playtime_at_review_min'] / 60
    band = pd.cut(hours, [0, 2, 10, 50, 100, 10**9], right=False, labels=['0~2h', '2~10h', '10~50h', '50~100h', '100h+'])
    result = reviews.groupby(band, observed=True)['voted_up'].agg(['size', 'mean'])
    result.columns = ['n', '긍정률']
    result['긍정률'] = (result['긍정률'] * 100).round(1)
    return result


def summary_of(appid):
    """전체 리뷰 수와 긍정률만 빠르게 확인 (요청 1번)."""
    r = session.get(f'{BASE}/appreviews/{appid}', params={'json': 1, 'language': 'all', 'purchase_type': 'all', 'num_per_page': 0}, timeout=30)
    q = r.json()['query_summary']
    return {'total': q['total_reviews'], 'positive_rate': round(q['total_positive'] / q['total_reviews'] * 100, 1)}

# 예: summary_of(1174180)  → RDR2 현재 전체 긍정률
# 예: positivity_by_playtime(pd.read_csv('starfield_reviews.csv'))  → Starfield 구간별 긍정률